# SVM Hardware Accelerator - FPGA Inference
### Breast Cancer Diagnostic - PYNQ-Z2 Deployment
---
**Instructions:**
1. Run **Cell 1** once to set up everything.
2. Use the **Upload CSV** button to select a patient record file.
3. Click **Run FPGA Inference** to get the diagnosis!


In [ ]:
# ============================================================
# CELL 1: Setup - Run this ONCE
# ============================================================
import time, os, sys, io
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from pynq import Overlay, allocate

# --- Scaler (zero-dependency, no joblib needed) ---
# These values were extracted from the trained MinMaxScaler on your laptop
MINS = np.array([6.981, 9.71, 43.79, 143.5, 0.05263, 0.01938, 0.0, 0.0, 0.106, 0.04996, 0.1115, 0.3602, 0.757, 6.802, 0.001713, 0.002252, 0.0, 0.0, 0.007882, 0.0008948, 7.93, 12.02, 50.41, 185.2, 0.07117, 0.02729, 0.0, 0.0, 0.1565, 0.05504])
MAXS = np.array([28.11, 39.28, 188.5, 2501.0, 0.1634, 0.3454, 0.4268, 0.2012, 0.304, 0.09744, 2.873, 4.885, 21.98, 542.2, 0.03113, 0.1354, 0.396, 0.05279, 0.07895, 0.02984, 36.04, 49.54, 251.2, 4254.0, 0.2226, 1.058, 1.252, 0.291, 0.6638, 0.2075])
RANGES = MAXS - MINS

def scale_features(raw):
    return (raw - MINS) / RANGES

# --- Load Bitstream ---
print('Loading SVM bitstream into FPGA fabric...')
overlay = Overlay('svm_final.bit')
dma     = overlay.axi_dma_0
svm_ip  = overlay.svm_top_0
print('Bitstream loaded successfully!')
print('Ready! Use the buttons below to run inference.')

In [ ]:
# ============================================================
# CELL 2: Interactive UI - Run this ONCE
# ============================================================

# --- Widgets ---
upload_btn   = widgets.FileUpload(accept='.csv', multiple=False, description='Upload CSV')
run_btn      = widgets.Button(description='Run FPGA Inference', button_style='success',
                              icon='bolt', layout=widgets.Layout(width='220px', height='40px'))
status_label = widgets.Label(value='Upload a patient_record CSV file, then click Run.')
output_area  = widgets.Output()

current_df = [None]  # Store loaded dataframe

# --- When a file is uploaded ---
def on_upload(change):
    if upload_btn.value:
        uploaded_file = list(upload_btn.value.values())[0]
        content = uploaded_file['content']
        current_df[0] = pd.read_csv(io.BytesIO(bytes(content)))
        status_label.value = f'File loaded: {list(upload_btn.value.keys())[0]} ({len(current_df[0].columns)} columns). Now click Run!'

upload_btn.observe(on_upload, names='value')

# --- When Run button is clicked ---
def on_run(b):
    with output_area:
        clear_output(wait=True)
        
        if current_df[0] is None:
            print('ERROR: Please upload a CSV file first!')
            return
        
        df = current_df[0]
        true_label = df['True_Diagnosis'].iloc[0] if 'True_Diagnosis' in df.columns else 'Unknown'
        raw = df.drop(columns=['True_Diagnosis']).values[0] if 'True_Diagnosis' in df.columns else df.values[0]
        
        print('Running 1000 FPGA Inferences for benchmarking...')
        
        # Prepare FPGA input
        X_scaled = scale_features(raw)
        X_fpga   = np.round(X_scaled * 255).clip(0, 255).astype(np.uint32)
        
        in_buffer = allocate(shape=(30,), dtype=np.uint32)
        np.copyto(in_buffer, X_fpga)
        
        # Run 1000 iterations for benchmarking
        times = []
        for _ in range(1000):
            t0 = time.perf_counter()
            svm_ip.write(0x00, 0b10)  # Reset
            svm_ip.write(0x00, 0b01)  # Start
            dma.sendchannel.transfer(in_buffer)
            dma.sendchannel.wait()
            timeout = 100000
            while (svm_ip.read(0x04) & 0x1) == 0 and timeout > 0:
                timeout -= 1
            times.append((time.perf_counter() - t0) * 1000)
        
        pred   = svm_ip.read(0x08)
        cycles = svm_ip.read(0x0C)
        svm_ip.write(0x00, 0)  # Clear
        in_buffer.freebuffer()
        
        # Results
        diagnosis = 'MALIGNANT' if pred == 1 else 'BENIGN'
        color     = 'red'       if pred == 1 else 'green'
        mean_lat  = np.mean(times)
        hw_us     = cycles * 0.01  # 100MHz clock -> 10ns per cycle
        throughput = 1000.0 / mean_lat
        
        display(HTML(f"""
        <div style='font-family:monospace; background:#1e1e2e; color:#cdd6f4; padding:20px; border-radius:10px; margin:10px 0;'>
          <h2 style='text-align:center; color:#89b4fa;'> FPGA Inference Results</h2>
          <hr style='border-color:#45475a;'>
          <h2 style='text-align:center; color:{color};'>Diagnosis: {diagnosis}</h2>
          <p style='text-align:center; color:#a6e3a1;'>Actual Truth: {true_label}</p>
          <hr style='border-color:#45475a;'>
          <table style='width:100%; border-collapse:collapse;'>
            <tr><td style='padding:6px; color:#89dceb;'>Hardware Clock Cycles</td><td style='padding:6px; color:#f5c2e7;'>{cycles} cycles</td></tr>
            <tr><td style='padding:6px; color:#89dceb;'>Pure Hardware Compute</td><td style='padding:6px; color:#f5c2e7;'>{hw_us:.4f} &mu;s ({hw_us*1000:.4f} ns)</td></tr>
            <tr><td style='padding:6px; color:#89dceb;'>Mean System Latency</td><td style='padding:6px; color:#f5c2e7;'>{mean_lat:.5f} ms (incl. DMA + Python)</td></tr>
            <tr><td style='padding:6px; color:#89dceb;'>System Throughput</td><td style='padding:6px; color:#f5c2e7;'>{throughput:.1f} inferences/second</td></tr>
            <tr><td style='padding:6px; color:#89dceb;'>Estimated Power Draw</td><td style='padding:6px; color:#f5c2e7;'>~2.0 Watts</td></tr>
          </table>
        </div>
        """))

run_btn.on_click(on_run)

# --- Display UI ---
display(HTML('<h3 style="color:#89b4fa;">SVM FPGA Inference - Breast Cancer Diagnostic</h3>'))
display(upload_btn)
display(status_label)
display(run_btn)
display(output_area)